# step-counter-increment — ex2: log every N steps using modulo gate after step counter tick

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `step-counter-increment`. Running the final beacon cell reports progress against the `Trainer: step counter increment` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Trainer: step counter increment` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`step-counter-increment`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "step-counter-increment"
DD_SUBTOPIC = "Trainer: step counter increment"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `log_every` interval logging — quick refresher

Logging every step is wasteful for long runs — wandb's free tier throttles around 50 log/s, and per-step disk I/O adds up. The canonical guard is an interval check `if self.step % log_every == 0`:
```python
self.optimizer.step()
self.optimizer.zero_grad()
self.step += 1                              # tick FIRST
if self.step % log_every == 0:              # then gate
    self.log({'step': self.step, 'loss': loss.item()})
```
Tick THEN gate so step 100 logs at step==100 (not 99). With `log_every=10` you get logs at steps 10, 20, 30, ..., 100, 110.

### Exercise 2 — log every N steps using modulo gate after step counter tick

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the `step % log_every == 0` interval-gate AFTER the step counter tick so logged step values are multiples of `log_every` (10, 20, 30...) rather than off-by-one (9, 19, 29...).
> Keywords: log-every, interval-logging, modulo-gate, step-counter
> ```

**KCs targeted:** `step-counter-increments-after-optimizer-step`, `log-every-N-steps-modulo-gate`

Implement `ex2_train_with_log_every(losses, log_every)`. A training-loop-like driver that ticks a step counter every batch but only LOGS every `log_every` steps.

1. Initialize `step = 0`, `log = []`.
2. For each loss in `losses`:
   - (Simulated update happens here — nothing to compute.)
   - `step += 1` — tick AFTER the simulated update.
   - If `step % log_every == 0`: `log.append((step, loss))`.
3. Return `(step, log)`.

Inputs:
- `losses`: list of per-step floats.
- `log_every`: int >= 1 — emit a log entry every this-many steps.

Output:
- `final_step`: int — equal to `len(losses)`.
- `log`: list of `(step, loss)` tuples, with `step` taking values `log_every, 2*log_every, 3*log_every, ...`.

**Critical:** the tick MUST happen BEFORE the modulo check, otherwise step 10 would log at step==9 (off-by-one).

In [ ]:
def ex2_train_with_log_every(losses: list, log_every: int) -> tuple:
    """Tick step every batch; log only when step % log_every == 0."""
    raise NotImplementedError()


def _test_ex2():
    # === log_every=10 with 100 losses → log at steps 10, 20, ..., 100 ===
    losses = [1.0 / (i + 1) for i in range(100)]
    final, log = ex2_train_with_log_every(losses, log_every=10)
    assert final == 100, f'final_step should be 100; got {final}'

    logged_steps = [s for s, _ in log]
    assert logged_steps == [10, 20, 30, 40, 50, 60, 70, 80, 90, 100], (
        f'expected step multiples of 10 up to 100; got {logged_steps}'
    )
    assert len(log) == 10, f'should log 10 times in 100-step run; got {len(log)}'

    # Loss values at the logged steps match losses[step-1].
    for step, loss in log:
        expected_loss = losses[step - 1]
        assert loss == expected_loss, (
            f'log at step {step} has loss {loss}; expected losses[{step-1}]={expected_loss}'
        )

    # === log_every=1 → log every step ===
    final2, log2 = ex2_train_with_log_every([0.5, 0.4, 0.3], log_every=1)
    assert final2 == 3
    assert [s for s, _ in log2] == [1, 2, 3], f'log_every=1 should log every step; got {log2}'

    # === log_every greater than n_losses → ZERO log entries ===
    final3, log3 = ex2_train_with_log_every([0.5, 0.4, 0.3], log_every=10)
    assert final3 == 3, 'step counter still ticks even if no log entries'
    assert log3 == [], f'log_every=10 with 3 steps should produce no logs; got {log3}'

    # === Edge: log_every exactly equals n_losses → exactly ONE log at end ===
    final4, log4 = ex2_train_with_log_every([0.1, 0.2, 0.3, 0.4, 0.5], log_every=5)
    assert final4 == 5
    assert log4 == [(5, 0.5)], f'log_every=5 with 5 steps should log exactly (5, 0.5); got {log4}'

    # === Off-by-one detector: 9-step run with log_every=3 logs at 3, 6, 9 NOT 2, 5, 8 ===
    final5, log5 = ex2_train_with_log_every([float(i) for i in range(9)], log_every=3)
    logged5 = [s for s, _ in log5]
    assert logged5 == [3, 6, 9], (
        f'log_every=3 should log at multiples of 3 (tick BEFORE check); got {logged5}. '
        f'If you got [2, 5, 8] you incremented step AFTER the modulo check.'
    )

    # === Empty losses → no logs, step=0 ===
    final6, log6 = ex2_train_with_log_every([], log_every=5)
    assert final6 == 0 and log6 == []
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def ex2_train_with_log_every(losses, log_every):
    step = 0
    log = []
    for loss in losses:
        step += 1
        if step % log_every == 0:
            log.append((step, loss))
    return step, log
```

**Why tick BEFORE the modulo gate.** Logging at step 10 should reflect the state AFTER 10 updates have happened — same logic as ex1 in this folder. Tick first, then check `step % log_every`. If you reverse the order you log at step==9 instead of step==10 — the silent off-by-one that confuses 'why does my log file say step 9 when I asked for log_every=10?'

**`log_every=1` is the no-throttle case.** Useful for short debug runs where you want every step. The modulo check still works (`step % 1 == 0` always True).

**Variant: log on epoch boundary AND interval.** Real trainers often combine `if step % log_every == 0 OR end_of_epoch`. The interval catches mid-epoch detail; the epoch boundary catches the final post-validate state. Both gates live AFTER the tick.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()